# 🔬 Official DSA-Mamba — Eye Image HB Estimation
**Repo:** [First-Ronin/DSA-Mamba](https://github.com/First-Ronin/DSA-Mamba)  
**Architecture:** VSSM with Encoder-Decoder, SS_Conv_SSM blocks, series decomposition (trend + remainder), cross-attention skip connections  
**Tasks:** ① Binary Classification (Anemic vs Non-Anemic) ② HB Regression (g/dL)

---


In [ ]:
# ── 0. Clone official DSA-Mamba & install deps (run once) ──────────────────
import subprocess, sys

subprocess.run(["git", "clone", "--depth=1",
                "https://github.com/First-Ronin/DSA-Mamba.git", "DSA-Mamba"], check=False)
sys.path.insert(0, "DSA-Mamba")

packages = [
    "mamba-ssm",
    "causal-conv1d",
    "einops",
    "timm",
    "pandas",
    "openpyxl",
    "scikit-learn",
    "matplotlib",
    "tqdm",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("✅ Done")


In [ ]:
# ── 1. Patch missing cross_attention.py into cloned repo ────────────────────
# The official repo is missing cross_attention.py source (only .pyc exists)
# We reconstruct it from bytecode analysis below.

import os

cross_attention_src = '''
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

class DWConv(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, groups=dim, bias=True)
    def forward(self, x):
        B, H, W, C = x.size()
        tx = x.permute(0, 3, 1, 2)
        conv_x = self.dwconv(tx)
        return conv_x.flatten(2).transpose(1,2).view(B, H, W, C)

class skip_ffn(nn.Module):
    def __init__(self, c1, c2):
        super().__init__()
        self.fc1=nn.Linear(c1,c2); self.dwconv=DWConv(c2); self.act=nn.GELU()
        self.fc2=nn.Linear(c2,c1); self.norm1=nn.LayerNorm(c2)
        self.norm2=nn.LayerNorm(c1); self.norm3=nn.LayerNorm(c1)
    def forward(self, x):
        ax=self.act(self.norm1(self.dwconv(self.fc1(x))))
        return self.norm3(self.fc2(ax)+x)

class Cross_Attention(nn.Module):
    def __init__(self, key_channels, value_channels, head_count=1):
        super().__init__()
        self.key_channels=key_channels; self.head_count=head_count; self.value_channels=value_channels
        self.reprojection=nn.Conv2d(value_channels, key_channels, kernel_size=1)
        self.norm=nn.LayerNorm(key_channels)
    def forward(self, x1, x2):
        B, H, W, C=x1.size()
        keys=x2.view(B,-1,C).transpose(1,2); queries=x1.view(B,-1,C).transpose(1,2); values=x2.view(B,-1,C).transpose(1,2)
        hk=self.key_channels//self.head_count; hv=self.value_channels//self.head_count
        attended=[]
        for i in range(self.head_count):
            k=F.softmax(keys[:,i*hk:(i+1)*hk,:],dim=2)
            q=F.softmax(queries[:,i*hk:(i+1)*hk,:],dim=1)
            v=values[:,i*hv:(i+1)*hv,:]
            ctx=torch.einsum("bdk,bvk->bdv",k,v)
            av=torch.einsum("bdv,bdl->bvl",ctx,q)
            attended.append(av)
        agg=torch.cat(attended,dim=1).reshape(B,-1,H,W)
        rep=self.reprojection(agg).permute(0,2,3,1)
        return self.norm(rep)

class CrossAttention(nn.Module):
    def __init__(self, in_dim, key_dim, value_dim, head_count=1, token_mlp=True):
        super().__init__()
        self.linear=nn.Linear(in_dim,key_dim); self.norm1=nn.LayerNorm(in_dim)
        self.attn=Cross_Attention(key_dim,value_dim,head_count)
        self.norm2=nn.LayerNorm(in_dim)
        self.mlp=skip_ffn(in_dim,int(in_dim*2)) if token_mlp else nn.Identity()
    def forward(self, x1, x2):
        n1=self.linear(self.norm1(x1)); n2=self.linear(self.norm1(x2))
        attn=self.attn(n1,n2)
        tx=attn+x1 if attn.shape==x1.shape else attn
        return self.mlp(self.norm2(tx)) if isinstance(self.mlp,skip_ffn) else tx
'''

model_dir = "DSA-Mamba/model"
ca_path   = os.path.join(model_dir, "cross_attention.py")
if not os.path.exists(ca_path):
    with open(ca_path, "w") as f:
        f.write(cross_attention_src)
    print(f"✅ cross_attention.py written to {ca_path}")
else:
    print(f"✅ cross_attention.py already exists")


In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import os, sys, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, mean_absolute_error
import matplotlib.pyplot as plt
from tqdm import tqdm
warnings.filterwarnings("ignore")

sys.path.insert(0, "DSA-Mamba")
from model.DSAmamba import VSSM as DSAMambaBackbone

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
CFG = dict(
    image_dir   = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye",
    csv_path    = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx",
    image_col   = "Patient ID",
    hb_col      = "HB",
    hb_thresh   = 12.0,          # g/dL — below = anemic

    # --- Training ---
    epochs      = 50,
    batch_size  = 4,
    lr          = 1e-4,
    cls_weight  = 1.0,           # CE loss weight
    reg_weight  = 0.5,           # MSE loss weight for HB regression
    val_split   = 0.2,
    seed        = 42,
    num_workers = 2,
)


In [ ]:
# ── 4. Dual-head DSA-Mamba model ──────────────────────────────────────────────
class DSAMambaDualHead(nn.Module):
    """
    Official DSA-Mamba (VSSM) with:
      - classification head  → (B, 2)
      - regression head      → (B, 1)   for HB prediction
    """
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = DSAMambaBackbone(
            in_chans=3,
            num_classes=0,          # disable original head
            in_depths=[2, 2, 4],
            out_depths=[2, 2],
            in_dims=[96, 192, 384],
            out_dims=[768, 384],
        )
        feat_dim = 384   # last decoder dim (out_dims[-1])

        self.cls_head = nn.Linear(feat_dim, num_classes)
        self.reg_head = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(feat_dim // 2, 1),
        )

    def forward(self, x):
        # Run encoder-decoder backbone
        feat = self.backbone.forward_backbone(x)         # (B, H', W', C)
        feat = feat.permute(0, 3, 1, 2)                  # (B, C, H', W')
        feat = self.backbone.avgpool(feat)                # (B, C, 1, 1)
        feat = torch.flatten(feat, start_dim=1)          # (B, C)

        return self.cls_head(feat), self.reg_head(feat)

model = DSAMambaDualHead(num_classes=2).to(DEVICE)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"DSA-Mamba (official) Dual-Head | Params: {total/1e6:.2f}M")


In [ ]:
# ── 5. Dataset ────────────────────────────────────────────────────────────────
class EyeHBDataset(Dataset):
    def __init__(self, df, image_dir, image_col, hb_col, hb_thresh, transform=None):
        self.df=df.reset_index(drop=True); self.image_dir=image_dir
        self.image_col=image_col; self.hb_col=hb_col
        self.hb_thresh=hb_thresh; self.transform=transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row=self.df.iloc[idx]; pid=str(row[self.image_col]); hb=float(row[self.hb_col])
        label = 0 if hb < self.hb_thresh else 1
        img_path = None
        for ext in [".jpg",".jpeg",".png",".bmp",""]:
            p = os.path.join(self.image_dir, pid+ext)
            if os.path.exists(p): img_path=p; break
        if img_path is None:
            raise FileNotFoundError(f"No image for Patient ID '{pid}'")
        img = Image.open(img_path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long), torch.tensor([[hb]], dtype=torch.float32)

T_train = transforms.Compose([
    transforms.Resize((256,256)), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])
T_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])

df = pd.read_excel(CFG["csv_path"])
print(f"Samples: {len(df)} | Anemic (HB<{CFG['hb_thresh']}): {(df[CFG['hb_col']]<CFG['hb_thresh']).sum()}")
print(df[[CFG['image_col'], CFG['hb_col']]].describe())

train_df, val_df = train_test_split(df, test_size=CFG["val_split"], random_state=CFG["seed"],
    stratify=(df[CFG["hb_col"]]<CFG["hb_thresh"]).astype(int))

train_loader = DataLoader(EyeHBDataset(train_df, CFG["image_dir"], CFG["image_col"],
    CFG["hb_col"], CFG["hb_thresh"], T_train),
    batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"], pin_memory=True)

val_loader = DataLoader(EyeHBDataset(val_df, CFG["image_dir"], CFG["image_col"],
    CFG["hb_col"], CFG["hb_thresh"], T_val),
    batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")


In [ ]:
# ── 6. Training setup ─────────────────────────────────────────────────────────
optimizer  = optim.Adam(model.parameters(), lr=CFG["lr"])
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
ce_loss    = nn.CrossEntropyLoss()
mse_loss   = nn.MSELoss()

def compute_loss(logits, hb_pred, labels, hb_true):
    cl = ce_loss(logits, labels)
    rl = mse_loss(hb_pred, hb_true)
    total = CFG["cls_weight"] * cl + CFG["reg_weight"] * rl
    return total, cl.item(), rl.item()


In [ ]:
# ── 7. Train & Validate ───────────────────────────────────────────────────────
history = {"train_loss":[], "val_loss":[], "val_acc":[], "val_auc":[], "val_mae":[]}
best_val_loss = float("inf")

for epoch in range(1, CFG["epochs"]+1):
    # ─ Train ─
    model.train()
    running_loss = 0.0
    for imgs, labels, hb_true in tqdm(train_loader, desc=f"Ep {epoch}/{CFG['epochs']}", leave=False):
        imgs, labels, hb_true = imgs.to(DEVICE), labels.to(DEVICE), hb_true.to(DEVICE)
        optimizer.zero_grad()
        logits, hb_pred = model(imgs)
        loss, _, _ = compute_loss(logits, hb_pred, labels, hb_true)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    avg_train = running_loss / len(train_loader)

    # ─ Validate ─
    model.eval()
    val_loss = correct = total = 0
    all_preds, all_labels, all_scores = [], [], []
    all_hb_pred, all_hb_true = [], []

    with torch.no_grad():
        for imgs, labels, hb_true in val_loader:
            imgs, labels, hb_true = imgs.to(DEVICE), labels.to(DEVICE), hb_true.to(DEVICE)
            logits, hb_pred = model(imgs)
            loss, _, _ = compute_loss(logits, hb_pred, labels, hb_true)
            val_loss  += loss.item()
            probs      = torch.softmax(logits, 1)
            preds      = logits.argmax(1)
            correct   += (preds == labels).sum().item()
            total     += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_scores.extend(probs[:,1].cpu().tolist())
            all_hb_pred.extend(hb_pred.cpu().squeeze().tolist())
            all_hb_true.extend(hb_true.cpu().squeeze().tolist())

    val_loss /= len(val_loader)
    val_acc   = correct / total
    try:   val_auc = roc_auc_score(all_labels, all_scores)
    except: val_auc = float("nan")
    val_mae = mean_absolute_error(all_hb_true, all_hb_pred)

    history["train_loss"].append(avg_train)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
    history["val_mae"].append(val_mae)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_dsamamba_official.pth")

    if epoch % 5 == 0 or epoch == 1:
        print(f"Ep {epoch:3d} | TL: {avg_train:.4f} | VL: {val_loss:.4f} "
              f"| Acc: {val_acc:.3f} | AUC: {val_auc:.3f} | MAE: {val_mae:.2f} g/dL")

print(f"\n✅ Training done. Best val loss: {best_val_loss:.4f}")


In [ ]:
# ── 8. Plot training curves ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
ep = range(1, len(history["train_loss"])+1)

axes[0].plot(ep, history["train_loss"], label="Train"); axes[0].plot(ep, history["val_loss"], label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(ep, history["val_acc"], color="green")
axes[1].set_title("Val Accuracy"); axes[1].set_ylim(0,1); axes[1].set_xlabel("Epoch")

axes[2].plot(ep, history["val_auc"], color="purple")
axes[2].set_title("Val AUC"); axes[2].set_ylim(0,1); axes[2].set_xlabel("Epoch")

axes[3].plot(ep, history["val_mae"], color="orange")
axes[3].set_title("Val MAE — HB (g/dL)"); axes[3].set_xlabel("Epoch")

plt.suptitle("DSA-Mamba (Official) — Eye Image HB Estimation", fontsize=14, y=1.02)
plt.tight_layout(); plt.savefig("dsamamba_training_curves.png", dpi=120, bbox_inches="tight"); plt.show()


In [ ]:
# ── 9. Final evaluation ───────────────────────────────────────────────────────
model.load_state_dict(torch.load("best_dsamamba_official.pth", map_location=DEVICE))
model.eval()

all_preds, all_labels, all_hbp, all_hbt, all_scores = [], [], [], [], []
with torch.no_grad():
    for imgs, labels, hb_true in val_loader:
        logits, hb_pred = model(imgs.to(DEVICE))
        probs = torch.softmax(logits, 1)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())
        all_scores.extend(probs[:,1].cpu().tolist())
        all_hbp.extend(hb_pred.cpu().squeeze().tolist())
        all_hbt.extend(hb_true.cpu().squeeze().tolist())

print("=" * 60)
print("CLASSIFICATION REPORT")
print(classification_report(all_labels, all_preds, target_names=["Anemic","Normal"]))
try:
    print(f"AUC      : {roc_auc_score(all_labels, all_scores):.4f}")
except: pass
print(f"HB MAE   : {mean_absolute_error(all_hbt, all_hbp):.4f} g/dL")
rmse = np.sqrt(np.mean((np.array(all_hbt)-np.array(all_hbp))**2))
print(f"HB RMSE  : {rmse:.4f} g/dL")

# Scatter: true vs predicted HB
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(all_hbt, all_hbp, alpha=0.5, c=all_labels, cmap="RdYlGn")
mn,mx = min(all_hbt), max(all_hbt)
axes[0].plot([mn,mx],[mn,mx],"r--",label="Perfect"); axes[0].legend()
axes[0].set_xlabel("True HB (g/dL)"); axes[0].set_ylabel("Predicted HB (g/dL)")
axes[0].set_title("HB Regression: True vs Predicted")

# Error distribution
errs = np.array(all_hbp) - np.array(all_hbt)
axes[1].hist(errs, bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--"); axes[1].set_xlabel("Prediction Error (g/dL)")
axes[1].set_title(f"Error Distribution (mean={errs.mean():.2f}, std={errs.std():.2f})")

plt.tight_layout(); plt.savefig("dsamamba_hb_results.png", dpi=120); plt.show()


In [ ]:
# ── 10. Alternatively: run via command line (on Kaggle) ───────────────────────
# python train_hb.py \
#   --train-dataset-path /kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye \
#   --csv-path /kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx \
#   --image-col "Patient ID" \
#   --hb-col "HB" \
#   --hb-threshold 12.0 \
#   --batch-size 4 \
#   --epochs 50

print("All done! Use the cells above OR run train_hb.py from command line.")
